#Hand's On

🚀 O Apache Spark é uma plataforma de processamento de dados distribuída e de código aberto, projetada para processar grandes volumes de dados de forma rápida e escalável.

#Instalando Bibliotecas PySpark

In [ ]:
# Instalando pacote PySpark
!pip install pyspark

In [ ]:
# Instalação de funções básicas para o Spark
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.functions import countDistinct

##Iniciando Ambiente Spark e Criando a sessão SparkSession

In [ ]:
spark = (
    SparkSession
    .builder
    .master("local[*]")
    .getOrCreate()
)

##Consultando versão do Spark

In [ ]:
print(spark.version)

4.0.3


#Importação de Bases.csv

In [ ]:
# ATENÇÃO: Antes de Executar -> Faça o upload dos arquivos.csv compartilhados no Git da FIAP (Material Extra)
df_clientes = spark.read.csv('clientes.csv', sep = ';', inferSchema = True, header = True)
df_vendas = spark.read.csv('vendas.csv', sep = ',', inferSchema = True, header = True)

In [ ]:
df_clientes.show()
df_vendas.show()

+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|cliente_id|                nome|               email|idade|              cidade| UF|     sexo|data_cadastro|
+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|         1|        Miguel Porto|eloah69@nasciment...|   51|Sao Bernardo do C...| SP| Feminino|   24/09/2020|
|         2|Gustavo Henrique ...|   ubarros@gmail.com|   62|            Londrina| PR| Feminino|   28/05/2024|
|         3|       Julia Correia|ana-lauramoraes@u...|   65|            Sao Luis| MA| Feminino|   08/01/2020|
|         4|        Pietra Sales| usilveira@ig.com.br|   45|              Recife| PE|Masculino|   22/05/2021|
|         5|    Guilherme Barros|davi-luizfarias@b...|   24|              Cuiaba| MT| Feminino|   20/04/2021|
|         6|Sr. Guilherme Rez...|cavalcantieloah@b...|   57|Sao Bernardo do C...| SP|Masculino|   13/08/2023|
|         

#Exemplo de RDD

In [ ]:
dados_clientes = [
    (1,"João Silva", "joao.silva@gmail.com", 34, "São Paulo", "SP"),
    (2,"Maria Souza", "maria.souza@gmail.com", 28, "Curitiba", "PR")
]

In [ ]:
rdd_clientes = spark.sparkContext.parallelize(dados_clientes)

In [ ]:
rdd_clientes.collect()

[(1, 'João Silva', 'joao.silva@gmail.com', 34, 'São Paulo', 'SP'),
 (2, 'Maria Souza', 'maria.souza@gmail.com', 28, 'Curitiba', 'PR')]

#Exploração Inicial das Bases importadas (Clientes e Vendas)

In [ ]:
df_clientes.printSchema()

root
 |-- cliente_id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- email: string (nullable = true)
 |-- idade: integer (nullable = true)
 |-- cidade: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- sexo: string (nullable = true)
 |-- data_cadastro: string (nullable = true)



In [ ]:
df_vendas.printSchema()

root
 |-- venda_id: integer (nullable = true)
 |-- data_venda: date (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- produto: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- valor_unitario: double (nullable = true)



In [ ]:
df_clientes.show(5)
df_vendas.show(5)

+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|cliente_id|                nome|               email|idade|              cidade| UF|     sexo|data_cadastro|
+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|         1|        Miguel Porto|eloah69@nasciment...|   51|Sao Bernardo do C...| SP| Feminino|   24/09/2020|
|         2|Gustavo Henrique ...|   ubarros@gmail.com|   62|            Londrina| PR| Feminino|   28/05/2024|
|         3|       Julia Correia|ana-lauramoraes@u...|   65|            Sao Luis| MA| Feminino|   08/01/2020|
|         4|        Pietra Sales| usilveira@ig.com.br|   45|              Recife| PE|Masculino|   22/05/2021|
|         5|    Guilherme Barros|davi-luizfarias@b...|   24|              Cuiaba| MT| Feminino|   20/04/2021|
+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
only showi

##Seleção de Colunas

In [ ]:
df_clientes.select('nome', 'cidade') \
        .distinct() \
        .orderBy('cidade', ascending=True) \
        .show(5)

+--------------------+-------+
|                nome| cidade|
+--------------------+-------+
|     Dra. Nina Cunha|Aracaju|
|    Vitória Carvalho|Aracaju|
|     Rafaela Peixoto|Aracaju|
|Sr. Guilherme Rez...|Aracaju|
|          Ryan Pires|Aracaju|
+--------------------+-------+
only showing top 5 rows


In [ ]:
df_clientes.select(countDistinct('cidade') \
        .alias("qtd_clientes")) \
        .show()

+------------+
|qtd_clientes|
+------------+
|          39|
+------------+



##Seleção de Colunas Definindo parâmetro do conjunto de colunas

In [ ]:
col_1 = list(set(df_clientes.columns) - {'email','idade','cliente_id'})

df_clientes2 = df_clientes \
        .select(*col_1) \
        .distinct() \
        .orderBy('cidade', ascending=True) \
        .show(5)

+---+-------+---------+-------------+------------------+
| UF| cidade|     sexo|data_cadastro|              nome|
+---+-------+---------+-------------+------------------+
| SE|Aracaju| Feminino|   28/02/2023|Dra. Yasmin Taboao|
| SE|Aracaju| Feminino|   04/11/2023|  Joaquim Ferreira|
| SE|Aracaju|Masculino|   20/12/2021|     Vitor da Cruz|
| SE|Aracaju| Feminino|   17/06/2022|   Rafaela Peixoto|
| SE|Aracaju|Masculino|   24/09/2020|  Vitória Carvalho|
+---+-------+---------+-------------+------------------+
only showing top 5 rows


##Condição de Filtro com Where

In [ ]:
df_clientes.where("idade >= 30").show()

+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|cliente_id|                nome|               email|idade|              cidade| UF|     sexo|data_cadastro|
+----------+--------------------+--------------------+-----+--------------------+---+---------+-------------+
|         1|        Miguel Porto|eloah69@nasciment...|   51|Sao Bernardo do C...| SP| Feminino|   24/09/2020|
|         2|Gustavo Henrique ...|   ubarros@gmail.com|   62|            Londrina| PR| Feminino|   28/05/2024|
|         3|       Julia Correia|ana-lauramoraes@u...|   65|            Sao Luis| MA| Feminino|   08/01/2020|
|         4|        Pietra Sales| usilveira@ig.com.br|   45|              Recife| PE|Masculino|   22/05/2021|
|         6|Sr. Guilherme Rez...|cavalcantieloah@b...|   57|Sao Bernardo do C...| SP|Masculino|   13/08/2023|
|         8|Luiz Fernando Car...|joao-guilhermemor...|   55|       Florianopolis| SC| Feminino|   29/04/2021|
|         

In [ ]:
df_clientes.where("idade between 0 and 18").show()

+----------+--------------------+--------------+-----+--------------+---+---------+-------------+
|cliente_id|                nome|         email|idade|        cidade| UF|     sexo|data_cadastro|
+----------+--------------------+--------------+-----+--------------+---+---------+-------------+
|       117|Sr. Emanuel da Co...|renan57@da.net|   18|      Sao Luis| MA|Masculino|   10/02/2020|
|       251|Sr. Emanuel da Co...|renan57@da.net|   18|Rio de Janeiro| RJ|Masculino|   14/09/2022|
|       276|Sr. Emanuel da Co...|renan57@da.net|   18|        Cuiaba| MT| Feminino|   10/09/2023|
|       794|Sr. Emanuel da Co...|renan57@da.net|   18|     Fortaleza| CE|Masculino|   30/06/2021|
+----------+--------------------+--------------+-----+--------------+---+---------+-------------+



##Removendo Registros Duplicados

In [ ]:
df_clientes.count()

1000

In [ ]:
df_vendas.count()

1500

In [ ]:
df_vendas = df_vendas.drop_duplicates()
df_clientes = df_clientes.drop_duplicates()

#Applied Analytics - Responda os questionamentos

In [ ]:
# 1 - Qual cidade gerou o maior valor total de vendas?
# 2 - Qual o produto mais vendido em quantidade e sua média de valor unitário?

In [ ]:
df_join = df_vendas.join(df_clientes, on='cliente_id', how='inner')
df_join.show(5)

+----------+--------+----------+-------+----------+--------------+-------------------+--------------------+-----+---------+---+---------+-------------+
|cliente_id|venda_id|data_venda|produto|quantidade|valor_unitario|               nome|               email|idade|   cidade| UF|     sexo|data_cadastro|
+----------+--------+----------+-------+----------+--------------+-------------------+--------------------+-----+---------+---+---------+-------------+
|        50|     785|2025-01-19| Tablet|         2|        580.64|    Bernardo Farias|       zpires@da.com|   49|  Jundiai| SP|Masculino|   10/10/2021|
|       154|     551|2025-01-04| Tablet|         4|       3969.83|      Julia Correia|ana-lauramoraes@u...|   65|   Recife| PE|Masculino|   09/07/2020|
|       154|     454|2025-01-29|Teclado|         4|        265.91|      Julia Correia|ana-lauramoraes@u...|   65|   Recife| PE|Masculino|   09/07/2020|
|       313|    1020|2025-02-20|Teclado|         1|       1757.54|João Vitor Monteiro|da

In [ ]:
# Pergunta 01 - Qual cidade gerou o maior valor total de vendas?
cidade_maior_venda = df_join \
      .groupBy('cidade') \
      .agg(F.round(F.sum(F.col('quantidade') * F.col('valor_unitario')),2).alias('total_vendas')) \
      .orderBy(F.desc ('total_vendas')) \
      .show(1)

+-------+------------+
| cidade|total_vendas|
+-------+------------+
|Niteroi|   502446.27|
+-------+------------+
only showing top 1 row


In [ ]:
# Pergunta 02 - Qual o produto mais vendido em quantidade e sua média de valor unitário?
produto_mais_vendido = df_vendas \
        .groupBy('produto') \
        .agg(
            F.sum('quantidade').alias('soma_quantidade'),
            F.round(F.avg('valor_unitario'),2).alias('media_valor_unitario')
        ) \
        .orderBy(F.desc('soma_quantidade')) \
        .show()

+--------------+---------------+--------------------+
|       produto|soma_quantidade|media_valor_unitario|
+--------------+---------------+--------------------+
|        Tablet|            886|             3100.82|
|    Impressora|            685|             2864.13|
|      Notebook|            633|             2648.59|
|       Teclado|            549|             1875.65|
|       Celular|            449|             2719.12|
|       Monitor|            438|             2769.29|
|Fone de Ouvido|            379|             2375.51|
|         Mouse|            183|             3570.61|
+--------------+---------------+--------------------+



#Testando consultas via SparkSQL

In [ ]:
# Criação de View para utilização no SparkSql
df_clientes.createOrReplaceTempView('clientes')
df_vendas.createOrReplaceTempView('vendas')

In [ ]:
spark.sql(
'''
SELECT CIDADE, COUNT(DISTINCT CLIENTE_ID) AS QTD_CLIENTES
FROM CLIENTES
GROUP BY CIDADE
ORDER BY QTD_CLIENTES DESC
'''
).show(5)

+-------------------+------------+
|             CIDADE|QTD_CLIENTES|
+-------------------+------------+
|             Santos|          40|
|            Niteroi|          38|
|           Sorocaba|          33|
|Sao Jose dos Campos|          33|
|           Salvador|          31|
+-------------------+------------+
only showing top 5 rows


In [ ]:
# Pergunta 03 - Top 5 Clientes com mais compras de produtos

spark.sql(
'''
SELECT DISTINCT
A.NOME AS NOME_CLIENTE,
SUM(B.QUANTIDADE) AS QTD_COMPRAS
FROM CLIENTES AS A
LEFT JOIN VENDAS AS B ON (A.CLIENTE_ID = B.CLIENTE_ID)
GROUP BY A.NOME
ORDER BY QTD_COMPRAS DESC
LIMIT 5
'''
).show()

+-------------------+-----------+
|       NOME_CLIENTE|QTD_COMPRAS|
+-------------------+-----------+
|       Calebe Nunes|        128|
|    Bernardo da Luz|         98|
|     Eduardo Campos|         98|
|João Vitor Monteiro|         88|
|         Luna Gomes|         85|
+-------------------+-----------+



In [ ]:
# Pergunta 04 - Qual foi a data com maior total de vendas
spark.sql(
'''
SELECT DATA_VENDA,
       ROUND(SUM(QUANTIDADE * VALOR_UNITARIO),2) AS TOTAL_VENDAS
FROM VENDAS
GROUP BY DATA_VENDA
ORDER BY TOTAL_VENDAS DESC
LIMIT 1
'''
).show()

+----------+------------+
|DATA_VENDA|TOTAL_VENDAS|
+----------+------------+
|2025-01-08|    661361.7|
+----------+------------+



#Renomeando Colunas

In [ ]:
df_final = (
    df_join
    .withColumnRenamed('quantidade', 'qtd_vendas')
    .withColumnRenamed('produto', 'nome_produto')

).show()

+----------+--------+----------+--------------+----------+--------------+--------------------+--------------------+-----+--------------+---+---------+-------------+
|cliente_id|venda_id|data_venda|  nome_produto|qtd_vendas|valor_unitario|                nome|               email|idade|        cidade| UF|     sexo|data_cadastro|
+----------+--------+----------+--------------+----------+--------------+--------------------+--------------------+-----+--------------+---+---------+-------------+
|        50|     785|2025-01-19|        Tablet|         2|        580.64|     Bernardo Farias|       zpires@da.com|   49|       Jundiai| SP|Masculino|   10/10/2021|
|       154|     551|2025-01-04|        Tablet|         4|       3969.83|       Julia Correia|ana-lauramoraes@u...|   65|        Recife| PE|Masculino|   09/07/2020|
|       154|     454|2025-01-29|       Teclado|         4|        265.91|       Julia Correia|ana-lauramoraes@u...|   65|        Recife| PE|Masculino|   09/07/2020|
|       31